In [ ]:
def stable_id(*parts):
    s = '||'.join([p or '' for p in parts])
    return hashlib.sha1(s.encode('utf-8')).hexdigest()[:16]

def ingest_web_evidence_for_step(step_text: str, num_results: int = 5, chunks_per_page: int = 6):
    results = google_cse_search(step_text, num=num_results)
    new_docs = []
    for r in results:
        url = r['link']
        title = r.get('title','')
        domain = r.get('displayLink','') or urlparse(url).netloc
        try:
            page_text = fetch_page_text(url)
        except Exception:
            continue
        chunks = chunk_text(page_text)[:chunks_per_page]
        for ci, ch in enumerate(chunks):
            doc_id = stable_id(url, str(ci), ch[:80])
            new_docs.append({
                'id': f'web_{doc_id}',
                'text': ch,
                'source': domain,
                'url': url,
                'title': title,
            })
    added = store.add_many(new_docs)
    return {'search_results': results, 'added_chunks': added}

def retrieve_step_evidence(step_text: str, k: int = 5, refresh_web: bool = True):
    meta = {}
    if refresh_web:
        meta = ingest_web_evidence_for_step(step_text)
    candidates = store.search(step_text, k=k)
    return candidates, meta


In [ ]:
STOPWORDS = set('''
a an the and or but if then else this that these those is are was were be been being
of to in for on with as by at from into about over under between within without
it its their them they we you your our i he she his her
'''.split())

def normalize_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r'[^a-z0-9\s\-]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def extract_keywords(s: str):
    toks = [t for t in normalize_text(s).split() if t not in STOPWORDS and len(t) > 2]
    return toks

def coverage_score(claim: str, evidence: str) -> float:
    ck = set(extract_keywords(claim))
    if not ck:
        return 0.0
    ek = set(extract_keywords(evidence))
    return len(ck & ek) / len(ck)

def compare_claim_to_candidates(claim: str, candidates: list, sim_threshold: float = 0.30, cov_threshold: float = 0.15):
    filtered = [c for c in candidates if c.get('similarity', 0.0) >= sim_threshold]
    if not filtered:
        return {'verdict': 'NO_EVIDENCE', 'best': None, 'candidates': []}
    for c in filtered:
        c['coverage'] = coverage_score(claim, c['text'])
    filtered.sort(key=lambda x: (x['similarity'], x['coverage']), reverse=True)
    best = filtered[0]
    if best['coverage'] >= cov_threshold:
        return {'verdict': 'SUPPORTED_WEAK', 'best': best, 'candidates': filtered}
    return {'verdict': 'WEAK', 'best': best, 'candidates': filtered}


In [ ]:
def verify_reasoning_with_rag_google(reasoning_json, k=5, refresh_web=True):
    results = []
    for step_obj in reasoning_json['reasoning']:
        claim = step_obj['claim']
        candidates, meta = retrieve_step_evidence(claim, k=k, refresh_web=refresh_web)
        comp = compare_claim_to_candidates(claim, candidates)
        results.append({
            'step': step_obj['step'],
            'claim': claim,
            'verdict': comp['verdict'],
            'best_evidence': comp['best'],
            'top_evidence': comp['candidates'][:k],
            'web_ingest_meta': meta,
        })
    return results

out = verify_reasoning_with_rag_google(reasoning_json, k=5, refresh_web=True)
out


[{'step': 1,
  'claim': 'The sky appears blue primarily due to Rayleigh scattering in Earth’s atmosphere.',
  'verdict': 'SUPPORTED_WEAK',
  'best_evidence': {'id': 'web_39f6ddcfd6d9a03d',
   'text': "William Strutt). [ 2 ] Due to Rayleigh scattering, red and orange colors are more visible during sunset because the blue and violet light has been scattered out of the direct path. Due to removal of such colors, these colors are scattered by dramatically colored skies and monochromatic rainbows . Rayleigh scattering results from the electric polarizability of the particles. The oscillating electric field of a light wave acts on the charges within a particle, causing them to move at the same frequency. The particle, therefore, becomes a small radiating dipole whose radiation we see as scattered light. The particles may be individual atoms or molecules; it can occur when light travels through transparent solids and liquids, but is most prominently seen in gases . Rayleigh scattering of sunl

In [ ]:
def show_google_and_ranking(out, top_n=5):
    for r in out:
        claim = r["claim"]
        meta = r.get("web_ingest_meta", {}) or {}
        google_hits = meta.get("search_results", []) or []
        ranked = r["top_evidence"] or []
        best = r["best_evidence"]

        print("\n" + "="*100)
        print(f"STEP {r['step']} CLAIM:\n{claim}")
        print(f"VERDICT: {r['verdict']} | Added web chunks: {meta.get('added_chunks', 0)}")
        print("-"*100)

        print("Google Search Results (Top URLs):")
        if not google_hits:
            print("  (No Google results returned)")
        else:
            for i, h in enumerate(google_hits[:5], start=1):
                print(f"  {i}. {h.get('title','')}")
                print(f"     {h.get('displayLink','')} | {h.get('link','')}")
                print(f"     Snippet: {h.get('snippet','')[:140]}")
        print("-"*100)

        print("Ranked Evidence Chunks (after embedding + FAISS search):")
        if not ranked:
            print("  (No evidence above thresholds)")
            continue

        for i, e in enumerate(ranked[:top_n], start=1):
            cov = e.get("coverage", coverage_score(claim, e["text"]))
            sim = e.get("similarity", 0.0)
            star = " ⭐BEST" if (best and e["id"] == best["id"]) else ""
            print(f"\n  Rank {i}{star}")
            print(f"    Similarity: {sim:.2f} | Coverage: {cov:.2f}")
            print(f"    Source: {e.get('source','')}")
            print(f"    Title : {e.get('title','')}")
            print(f"    URL   : {e.get('url','')}")
            print(f"    Text  : {e.get('text','')[:350]} ...")

show_google_and_ranking(out, top_n=5)



STEP 1 CLAIM:
The sky appears blue primarily due to Rayleigh scattering in Earth’s atmosphere.
VERDICT: SUPPORTED_WEAK | Added web chunks: 12
----------------------------------------------------------------------------------------------------
Google Search Results (Top URLs):
  1. Rayleigh scattering - Wikipedia
     en.wikipedia.org | https://en.wikipedia.org/wiki/Rayleigh_scattering
     Snippet: The phenomenon is named after the 19th-century British physicist Lord Rayleigh (John William Strutt). Rayleigh scattering causes the blue co
  2. Rayleigh Scattering - an overview | ScienceDirect Topics
     www.sciencedirect.com | https://www.sciencedirect.com/topics/physics-and-astronomy/rayleigh-scattering
     Snippet: ... of the isothermal compressibility βT. The blue of the sky. The blue of the sky is primarily due to the Rayleigh scattering of sunlight f
  3. Atmosphere of Earth - Wikipedia
     en.wikipedia.org | https://en.wikipedia.org/wiki/Atmosphere_of_Earth
     Snippet: Earth'

In [ ]:
def rag_hit_miss_evaluation(out):
    hits = 0
    for r in out:
        if r["verdict"] in ("SUPPORTED_WEAK",):
            hits += 1

    total = len(out)
    hit_rate = hits / total if total else 0.0

    print("\nRAG Retrieval Evaluation (Role 2)")
    print("--------------------------------")
    print(f"Steps evaluated : {total}")
    print(f"Hits            : {hits}")
    print(f"Misses          : {total - hits}")
    print(f"Hit Rate        : {hit_rate:.2f}")

rag_hit_miss_evaluation(out)



RAG Retrieval Evaluation (Role 2)
--------------------------------
Steps evaluated : 4
Hits            : 4
Misses          : 0
Hit Rate        : 1.00
